# Eval Mutu Retrieval

Menjalankan chatbot varian lokal end-to-end di Colab, lalu menilai mutu retrieval-nya
dengan DeepEval.

| Metrik | Menjawab pertanyaan | Butuh juri? |
|---|---|---|
| ContextualRelevancy | Berapa bagian potongan yang benar-benar relevan? | Ya |
| ContextualRecall | Informasi yang dibutuhkan acuan berhasil terambil? | Ya |
| ContextualPrecision | Potongan relevan berperingkat lebih tinggi? | Ya |
| section_hit_rate | Potongan berasal dari seksi FAQ yang ditunjuk acuan? | Tidak |

Mutu teks jawaban akhir tidak dinilai di sini. Yang menjaganya tetap `must_not_contain`
di dataset dan jaring pengaman eskalasi di kode.

Model yang **diuji** `qwen3:1.7b`, berjalan lokal di GPU Colab. Model **juri**
`claude-haiku-4-5` lewat Claude API. Keduanya berbeda dengan sengaja: model yang menilai
jawabannya sendiri bukan pengukuran, dan juri kecil menilai terlalu berisik untuk
dipercaya.

Juri adalah satu-satunya bagian yang memakai API berbayar. Chatbot yang diukur tetap
berjalan penuh di lokal tanpa kunci API — itu justru klaim yang sedang diuji notebook ini.

## Prasyarat

1. Runtime GPU T4 — `Runtime > Change runtime type > T4 GPU`
2. Colab Secrets (ikon kunci di panel kiri) berisi:
   - `SUPABASE_URL` dan `SUPABASE_SERVICE_KEY`
   - `ANTHROPIC_API_KEY` untuk juri

Jangan pernah menulis kredensial di dalam sel. Repo ini publik.

## 1. Ambil kode

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

!git clone --branch chatbot-local-llm --depth 1 https://github.com/AryaSeptiaputra/rag-fashion-chatbot.git /content/rag-fashion-chatbot
%cd /content/rag-fashion-chatbot

## 2. Pasang dependency

Memasang chromadb, llama-index, dan DeepEval di atas paket bawaan Colab.
Perlu beberapa menit.

`!pip install` yang gagal tidak menghentikan notebook: resolusi yang mentok
membuat seluruh berkas requirements batal dipasang -- termasuk `-e .` -- dan
errornya baru muncul jauh di bawah sebagai `ModuleNotFoundError: No module named
'app'`. Karena itu pip dijalankan lewat `subprocess.run` dengan status keluarnya
diperiksa.

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-eval.txt"],
    capture_output=True,
    text=True,
)
print(result.stdout, end="")
print(result.stderr, end="")
result.check_returncode()

## 3. Restart runtime

**Wajib.** Instalasi di atas mengganti numpy dan protobuf yang sudah dimuat Colab saat
runtime dinyalakan; tanpa restart, impor berikutnya gagal dengan pesan yang tidak
menunjuk penyebabnya.

Jalankan sel di bawah, tunggu runtime hidup lagi, lalu **lanjutkan dari sel 4** —
jangan mengulang dari atas.

In [ ]:
import IPython

IPython.get_ipython().kernel.do_shutdown(restart=True)

## 4. Nyalakan Ollama dan tarik model

Dijalankan setelah restart karena proses anak yang dimulai sebelum restart ikut mati.

`install.sh` sengaja tidak dipakai di sini. Di Colab dua bagiannya mubazir --
service systemd (server dijalankan manual di sel berikutnya) dan pemasangan driver
NVIDIA (sudah ada di runtime) -- sementara bagian yang dibutuhkan justru rapuh:

1. Arsip rilis sekarang berformat `.tar.zst` dan butuh binary `zstd` yang tidak ada
   di image Colab. Tanpa itu installer berhenti dengan
   `ERROR: This version requires zstd for extraction`.
2. Installer menyalurkan `curl` langsung ke `tar` untuk arsip berukuran ~1,4 GB.
   Unduhan yang putus di tengah membuat `tar` berhenti dengan status 2, dan pesan
   aslinya tenggelam di dalam pipeline.

Jadi arsipnya diunduh ke berkas lebih dulu (bisa diulang oleh `--retry` tanpa
menyentuh tahap ekstraksi), baru diekstrak. Tiap langkah mencetak keluarannya
sendiri sebelum status keluarnya diperiksa, supaya kegagalan terbaca di selnya,
bukan muncul beberapa sel kemudian sebagai `ollama` yang tidak ditemukan.

In [ ]:
%cd /content/rag-fashion-chatbot

import subprocess
from pathlib import Path

ARCHIVE_URL = "https://ollama.com/download/ollama-linux-amd64.tar.zst"
ARCHIVE_PATH = Path("/tmp/ollama-linux-amd64.tar.zst")


def run(command: list[str]) -> None:
    """Jalankan perintah dan cetak keluarannya sebelum memeriksa status keluar.

    Keluaran subprocess tidak ikut tertangkap sel Colab kalau dibiarkan
    mewarisi file descriptor kernel, jadi ditangkap dan dicetak eksplisit.

    Args:
        command: Perintah beserta argumennya.

    Raises:
        subprocess.CalledProcessError: Kalau status keluarnya bukan nol.
    """
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="")
    result.check_returncode()


run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "zstd"])

# Unduhan senyap (-sS) supaya progress meter tidak membanjiri keluaran sel.
# Berkas lama dihapus lebih dulu karena sisa unduhan yang putus akan lolos -f.
ARCHIVE_PATH.unlink(missing_ok=True)
run([
    "curl", "-fL", "-sS",
    "--retry", "5", "--retry-delay", "2", "--retry-all-errors",
    "-o", str(ARCHIVE_PATH), ARCHIVE_URL,
])
print(f"arsip terunduh: {ARCHIVE_PATH.stat().st_size / 1024 ** 3:.2f} GiB")

# Arsip tidak punya folder tingkat atas -- isinya langsung bin/ dan lib/ --
# jadi diekstrak ke /usr/local supaya binary mendarat di /usr/local/bin/ollama.
run(["tar", "--zstd", "-xf", str(ARCHIVE_PATH), "-C", "/usr/local"])

In [ ]:
import os
import shutil
import subprocess

# Installer menaruh binary di /usr/local/bin. PATH kernel Colab tidak selalu
# memuatnya sampai runtime dimulai ulang, jadi ditambahkan di sini.
os.environ["PATH"] = os.environ["PATH"] + ":/usr/local/bin"

OLLAMA = shutil.which("ollama")
if OLLAMA is None:
    raise RuntimeError(
        "Binary ollama tidak ditemukan. Jalankan ulang sel instalasi di atas dan "
        "baca keluarannya sampai habis -- sel itu mencetak keluaran tiap langkah "
        "sebelum memeriksa status keluarnya."
    )

print(subprocess.run([OLLAMA, "--version"], capture_output=True, text=True).stdout)

In [ ]:
import subprocess
import time
from pathlib import Path

import httpx

subprocess.Popen(
    [OLLAMA, "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)


def wait_for_ollama(timeout_seconds: int = 60) -> None:
    """Tunggu server Ollama siap menerima permintaan.

    Args:
        timeout_seconds: Batas tunggu sebelum menyerah.

    Raises:
        RuntimeError: Kalau server tidak merespons sampai batas waktu.
    """
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            httpx.get("http://localhost:11434/api/version", timeout=2.0)
            print("Ollama siap")
            return
        except httpx.HTTPError:
            time.sleep(2)

    log = Path("/content/ollama.log").read_text(encoding="utf-8", errors="replace")
    raise RuntimeError(
        f"Ollama tidak merespons dalam {timeout_seconds} detik. "
        f"Isi /content/ollama.log:\n{log[-2000:]}"
    )


wait_for_ollama()

In [ ]:
# Hanya model yang diuji yang ditarik. Juri berjalan di Claude API, dan metrik
# DeepEval tidak memakai embedding sama sekali.
!{OLLAMA} pull qwen3:1.7b
!{OLLAMA} list

## 5. Kredensial dan konfigurasi

Env var harus diset **sebelum** `app` di-import pertama kali: `app.config.settings`
adalah singleton tingkat modul yang membaca environment sekali saat impor.

Tiga kredensial diambil dari panel **Secrets** Colab (ikon kunci di sidebar kiri).
Untuk tiap nama -- `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`, `ANTHROPIC_API_KEY` --
tekan *Add new secret*, isi kolom Name persis seperti itu, tempel nilainya di Value,
lalu nyalakan toggle *Notebook access*. Toggle itu berlaku per notebook: secret yang
sudah ada pun tidak terbaca kalau toggle-nya mati untuk notebook ini.

Kalau panel Secrets tidak bisa dihubungi, selnya jatuh ke prompt `getpass` supaya
eval tetap jalan. Nilai yang dimasukkan lewat prompt hanya hidup selama sesi runtime
dan tidak ikut tersimpan di notebook.

In [ ]:
import os
from getpass import getpass

from google.colab import errors, userdata

SECRET_NAMES = ("SUPABASE_URL", "SUPABASE_SERVICE_KEY", "ANTHROPIC_API_KEY")


def load_secret(name: str) -> str:
    """Ambil kredensial dari Colab Secrets, jatuh ke prompt manual kalau gagal.

    Args:
        name: Nama secret di panel Secrets Colab.

    Returns:
        Nilai kredensial.
    """
    try:
        return userdata.get(name)
    except (errors.Error, userdata.NotebookAccessError) as error:
        # errors.Error mencakup SecretNotFoundError, TimeoutException, dan
        # MessageError -- yang terakhir dilempar saat panggilan HTTP ke endpoint
        # secrets gagal (mis. 401 karena sesi browser tidak terautentikasi).
        print(f"{name}: Colab Secrets tidak terbaca ({error})")
        return getpass(f"Tempel {name}: ")


for secret_name in SECRET_NAMES:
    os.environ[secret_name] = load_secret(secret_name)

os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"
os.environ["LLM_MODEL"] = "qwen3:1.7b"
os.environ["COMPOSER_MODEL"] = "qwen3:1.7b"
os.environ["JUDGE_MODEL"] = "claude-haiku-4-5"
os.environ["EMBEDDING_DEVICE"] = "cuda"

from app.config import settings

print("model diuji :", settings.llm_model, "/", settings.composer_model, "(lokal)")
print("model juri  :", settings.judge_model, "(Claude API)")
print("supabase    :", "terisi" if settings.supabase_url else "KOSONG")
print("kunci juri  :", "terisi" if settings.anthropic_api_key else "KOSONG")

## 6. Bangun index FAQ

Vector store Colab kosong setiap sesi, jadi FAQ di-ingest ulang dari `data/raw/faq/`.
Unduhan model embedding (~450 MB) terjadi sekali.

In [ ]:
!python scripts/ingest_faq.py

## 7. Jalankan kasus eval

Dua dataset dijalankan terpisah:

- `evals/dataset.jsonl` — 41 kasus, seluruh permukaan 8 tool. Menghasilkan **akurasi
  pemilihan tool**, angka yang mengisi tabel perbandingan di README. Tidak dinilai juri.
- `evals/retrieval.jsonl` — 16 kasus FAQ dengan acuan. Sumber metrik **mutu retrieval**.

Keduanya berjalan penuh di GPU Colab dan tidak berbiaya. Trace ditulis ke disk sebelum
penilaian dimulai, jadi kalau sesi putus saat menilai, tahap ini tidak perlu diulang.

In [ ]:
!python scripts/run_eval.py \
    --dataset evals/dataset.jsonl \
    --output outputs/eval_report_tools.json \
    --trace-output outputs/trace_tools.jsonl

In [ ]:
!python scripts/run_eval.py \
    --dataset evals/retrieval.jsonl \
    --output outputs/eval_report_retrieval.json \
    --trace-output outputs/trace_retrieval.jsonl

## 8. Nilai mutu retrieval (DeepEval)

Selain tiga metrik yang dinilai LLM, `section_hit_rate` dihitung deterministik: berapa
bagian potongan yang benar-benar berasal dari seksi FAQ yang ditunjuk acuan. Metrik itu
tidak bergantung pada juri sama sekali, jadi ia tetap bisa dipercaya justru saat skor
juri terlihat mencurigakan.

In [ ]:
from pathlib import Path

from app.evals.judge import build_deepeval_judge
from app.evals.retrieval import RetrievalQualityEvaluator
from app.evals.trace import FaqSectionIndex, load_traces

retrieval_report = RetrievalQualityEvaluator(
    judge=build_deepeval_judge(),
    section_index=FaqSectionIndex.from_directory(),
).evaluate(load_traces(Path("outputs/trace_retrieval.jsonl")))

print("juri:", retrieval_report.judge_model)
print("agregat:", retrieval_report.aggregates)
print("gagal dinilai:", retrieval_report.unscored_total)
for note in retrieval_report.notes:
    print("catatan:", note)

## 9. Laporan

In [ ]:
import json

report_path = Path("outputs/quality_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(
    json.dumps(
        {
            "system_under_test": {
                "agent_model": settings.llm_model,
                "composer_model": settings.composer_model,
            },
            "retrieval_quality": retrieval_report.model_dump(),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Laporan lengkap disimpan di {report_path}")

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    [
        {"metrik": metric, "skor": score}
        for metric, score in retrieval_report.aggregates.items()
    ]
)
summary

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(9, 4))
axis.barh(summary["metrik"], summary["skor"])
axis.set_xlim(0, 1)
axis.set_xlabel(f"skor (juri: {retrieval_report.judge_model})")
axis.set_title(f"Mutu retrieval {settings.llm_model}")
axis.bar_label(axis.containers[0], fmt="%.2f", padding=3)
figure.tight_layout()
plt.show()

### Kasus terburuk

Rata-rata tanpa daftar kasus terburuk tidak bisa ditindaklanjuti: angka turun tanpa
petunjuk pertanyaan mana yang harus diperbaiki.

In [ ]:
for metric in retrieval_report.aggregates:
    print()
    print(metric)
    for case in retrieval_report.worst_cases(metric, limit=3):
        print(f"  {case.scores[metric]:.2f}  [{case.case_id}] {case.question}")

## Cara membaca angkanya

**Juri dan yang diuji harus selalu disebut berpasangan.** Angka di sini berarti
"Qwen3-1.7B lokal, dinilai oleh `claude-haiku-4-5`". Mengganti salah satunya membuat
angkanya tidak lagi sebanding dengan run sebelumnya, jadi `judge_model` ikut tersimpan
di laporan — jangan mencatat skornya terpisah dari nama jurinya.

**Juri yang kuat memindahkan sumber keraguan, bukan menghapusnya.** Dengan Haiku 4.5
skor rendah lebih mungkin benar-benar berasal dari retrieval, bukan dari juri yang
bingung. Yang tersisa: penilaian relevansi tetap bergantung pada tafsir juri atas
"relevan", dan potongan yang benar tapi ditulis dengan kata berbeda dari pertanyaannya
kadang dihukum.

**Mutu teks jawaban tidak diukur di sini.** Yang menjaganya `must_not_contain` di
dataset dan jaring pengaman eskalasi di kode. Retrieval yang baik tidak menjamin
jawaban akhirnya juga baik.

**Korpus FAQ hanya 9 potongan dari satu dokumen contoh.** Metrik retrieval di atas 16
pertanyaan membuktikan pipeline-nya bekerja, bukan bahwa retrieval-nya bagus pada
korpus produksi. Ganti `data/raw/faq/` dengan dokumen asli sebelum menarik kesimpulan
soal kualitas.

**Gagal-nilai bukan skor nol.** `unscored_total` menghitung kasus yang jurinya gagal
mengeluarkan JSON sesuai skema. Kasus itu tidak ikut rata-rata. Angka yang besar di
sini menunjuk ke masalah transport atau rate limit, bukan ke mutu chatbot.

**Hanya sel penilaian yang memanggil API berbayar.** 16 kasus retrieval dikali tiga
metrik, dan tiap metrik memakai lebih dari satu panggilan. Seluruh tahap generate — 57
kasus — tidak berbiaya karena berjalan penuh di GPU Colab. Jadi iterasi prompt dan
chunking sebaiknya memakai akurasi tool lebih dulu, dan penilaian retrieval dijalankan
saat ada yang benar-benar ingin diukur.

**`section_hit_rate` tidak memakai juri.** Kalau metrik DeepEval terlihat buruk
sementara `section_hit_rate` tinggi, kecurigaan pertama harus jatuh ke jurinya.